# Merge BERTopic Models Across Outlets

This notebook loads the saved BERTopic models for Tagesschau, RT, Antispiegel, Tichys Einblick, Nius, Compact, and Deutschlandkurier, merges them with `BERTopic.merge_models(...)`, then builds merged article-level UMAP maps.


In [ ]:
import os
import sys
from pathlib import Path

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.7


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "1a_BERTopic" / "local_outputs",
    PROJECT_ROOT / "1a_BERTopic" / "outputs",
    PROJECT_ROOT / "BERTopic" / "outputs",
]
MERGED_SAVE_DIR = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_all_outlets_model"

print(f"Project root: {PROJECT_ROOT}")
print("Model path candidates:")
for candidate in MODEL_DIR_CANDIDATES:
    print(f"  - {candidate}")


In [ ]:
import importlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from bertopic import BERTopic

import merged_outlets_analysis as moa
moa = importlib.reload(moa)

ALT_MEDIA_OUTLET_KEYS = getattr(
    moa,
    "ALT_MEDIA_OUTLET_KEYS",
    ("rt", "antispiegel", "tichys", "nius", "compact", "deutschlandkurier"),
)
OUTLET_SPECS = moa.OUTLET_SPECS
build_merged_article_frame = moa.build_merged_article_frame
build_outlet_topic_coverage_summary = moa.build_outlet_topic_coverage_summary
combine_prepared_documents = moa.combine_prepared_documents
load_all_prepared_documents = moa.load_all_prepared_documents
plot_merged_topic_umap = moa.plot_merged_topic_umap
plot_outlet_highlight_umap = moa.plot_outlet_highlight_umap
resolve_model_paths = moa.resolve_model_paths

MODEL_PATHS = resolve_model_paths(MODEL_DIR_CANDIDATES)

loaded_models = {
    key: BERTopic.load(model_path, embedding_model=EMBEDDING_MODEL)
    for key, model_path in MODEL_PATHS.items()
}

for key, model in loaded_models.items():
    topic_info = model.get_topic_info()
    print(f"Loaded {key}: {MODEL_PATHS[key]} ({len(topic_info)} rows in topic info)")

tm_ts = loaded_models["tagesschau"]
tm_rt = loaded_models["rt"]
tm_as = loaded_models["antispiegel"]
tm_te = loaded_models["tichys"]
tm_ns = loaded_models["nius"]
tm_cm = loaded_models["compact"]
tm_dk = loaded_models["deutschlandkurier"]


In [ ]:
models_to_merge = [
    tm_ts,
    tm_rt,
    tm_as,
    tm_te,
    tm_ns,
    tm_cm,
    tm_dk,
]

merged_model = BERTopic.merge_models(
    models_to_merge,
    min_similarity=MIN_SIMILARITY,
    embedding_model=EMBEDDING_MODEL,
)

merged_topic_info = merged_model.get_topic_info()
display(merged_topic_info.head(30))
print("Merged topic count:", len(merged_topic_info))


In [ ]:
import shutil

SAVE_MERGED_MODEL = False

if SAVE_MERGED_MODEL:
    MERGED_SAVE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if MERGED_SAVE_DIR.exists():
        shutil.rmtree(MERGED_SAVE_DIR)
    merged_model.save(
        MERGED_SAVE_DIR,
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=EMBEDDING_MODEL,
    )
    print(f"Saved merged model to: {MERGED_SAVE_DIR}")
else:
    print("Skipped save. Set SAVE_MERGED_MODEL = True to persist the merged model.")


In [ ]:
import pandas as pd

prepared_by_outlet = load_all_prepared_documents(PROJECT_ROOT)
prepared_summary = pd.DataFrame(
    [
        {
            "Outlet": OUTLET_SPECS[key].label,
            "Prepared_Documents": len(df),
        }
        for key, df in prepared_by_outlet.items()
    ]
).sort_values("Outlet").reset_index(drop=True)
display(prepared_summary)

combined_prepared = combine_prepared_documents(prepared_by_outlet)
print("Combined prepared documents:", len(combined_prepared))


In [ ]:
merged_articles, merged_topic_info_display, merged_umap_model = build_merged_article_frame(
    merged_model,
    combined_prepared,
)

display(merged_articles[["outlet_label", "document_id", "merged_topic", "merged_display_label"]].head())
display(merged_articles.groupby("outlet_label").size().rename("Article_Count").reset_index())


In [ ]:
fig, ax = plot_merged_topic_umap(
    merged_articles,
    merged_topic_info_display,
    top_n=20,
)
plt.show()


In [ ]:
coverage_summary = build_outlet_topic_coverage_summary(
    merged_articles,
    merged_topic_info_display,
)
display(coverage_summary)

for outlet_key in ALT_MEDIA_OUTLET_KEYS:
    spec = OUTLET_SPECS[outlet_key]
    print(f"Plotting alternative-media outlet coverage: {spec.label}")
    fig, ax = plot_outlet_highlight_umap(
        merged_articles,
        outlet_key,
        alt_media_only=False,
        show_kde=True,
        min_label_articles=15,
        merged_topic_info=merged_topic_info_display,
    )
    plt.show()
